In [31]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [32]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O') or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    with open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg",'rb') as f:
         file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg", "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH,WW=img.shape[:2]
    cropped=img[0:200,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [35]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df):
    for n in range(0,len(df)):
        #Extract key info of office
        Row  = df.iloc[n]

        ###Collect Location information###
        Dept=Row["Dept"]
        Office=Row["Office"]
        print('['+str(n)+',"'+Dept+'","'+Office+'"]')
        try:        
            StrPage=dta[Year][Dept][Office]['Starting_Page']
            EndPage=dta[Year][Dept][Office]['Ending_Page']

            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb')
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            for Page in range(StrPage+1,EndPage+1):
                path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
                stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb') 
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                newimg = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)[0:200,0:WW]
                oldimg=np.concatenate((newimg,oldimg), axis=1)
            oldimg=cv2.copyMakeBorder(oldimg,20,20,20,20,cv2.BORDER_CONSTANT)

            StrLoc=int(dta[Year][Dept][Office]['Office_X1'])
            EndLoc=int(dta[Year][Dept][Office]['Office_X2'])
            StrPage=int(dta[Year][Dept][Office]['Starting_Page'])
            EndPage=int(dta[Year][Dept][Office]['Ending_Page'])

            #Start#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            #End#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+"Page"+"{:03d}".format(EndPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(EndLoc,0),(EndLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            continue
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [63]:
def Show(n,StrLoc):
    Row=df.iloc[n]
    Page=Row['Page']
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg",'rb')
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    
    HH,WW=img.shape[:2]
    cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)
    cv2.imshow('Projection',img)
    cv2.waitKey(0)

In [36]:
Year='1938'
Showa='13'
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

In [37]:
n=0
Row  = df.iloc[n]
Page=int(Row["Page"])
Office=Row["Office"]
Dept=Row['Dept']
print(Office,Dept)
dta[Year][Dept][Office]={}
dta[Year][Dept][Office]["Starting_Page"]=Page

img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
#Convert to json via CLOVA
Json=Clova(Year,Page)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
HH=img.shape[:2][0]
print(Office+ 'success')

秘書課 Admin
秘書課success


In [38]:
#Test code| Version 2#
#Show Working office#
FailedList=[]
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["Starting_Page"]=Page
    print(dta[Year][Dept][Office])

    ###Collect Location information###
    ##Read image for first page##
    img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    #Convert to json via CLOVA
    Json=Clova(Year,Page)

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["Office_X1"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        HH=img.shape[:2][0]
        print(Office+ 'success: at row'+str(n))

{'Starting_Page': 2}
人事課success: at row1
{'Starting_Page': 4}
文書課success: at row2
{'Starting_Page': 6}
庶務課success: at row3
{'Starting_Page': 7}
企画課success: at row4
{'Starting_Page': 8}
財務課success: at row5
{'Starting_Page': 9}
統計課success: at row6
{'Starting_Page': 20}
都市計画課failed
{'Starting_Page': 24}
会計課success: at row8
{'Starting_Page': 26}
公債課success: at row9
{'Starting_Page': 27}
主税課success: at row10
{'Starting_Page': 34}
徴収課failed
{'Starting_Page': 39}
用度課success: at row12
{'Starting_Page': 43}
地理課success: at row13
{'Starting_Page': 48}
庶務課success: at row14
{'Starting_Page': 49}
商工課success: at row15
{'Starting_Page': 52}
貿易課success: at row16
{'Starting_Page': 53}
北京出張所success: at row17
{'Starting_Page': 54}
農漁課success: at row18
{'Starting_Page': 56}
権度課success: at row19
{'Starting_Page': 57}
庶務課success: at row20
{'Starting_Page': 58}
学務課success: at row21
{'Starting_Page': 60}
社会教育課failed
{'Starting_Page': 66}
体育課success: at row23
{'Starting_Page': 69}
視学課success: at row24
{'Startin

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\1938\\Page314\\Page314.jpg'

In [42]:
Check(df)

[0,"Admin","秘書課"]
[1,"Admin","人事課"]
[2,"Admin","文書課"]
[3,"企画局","庶務課"]
[4,"企画局","企画課"]
[5,"企画局","財務課"]
[6,"企画局","統計課"]
[7,"企画局","都市計画課"]
[8,"経理局","会計課"]
[9,"経理局","公債課"]
[10,"経理局","主税課"]
[11,"経理局","徴収課"]
[12,"経理局","用度課"]
[13,"経理局","地理課"]
[14,"産業局","庶務課"]
[15,"産業局","商工課"]
[16,"産業局","貿易課"]
[17,"産業局","北京出張所"]
[18,"産業局","農漁課"]
[19,"産業局","権度課"]
[20,"教育局","庶務課"]
[21,"教育局","学務課"]
[22,"教育局","社会教育課"]
[23,"教育局","体育課"]
[24,"教育局","視学課"]
[25,"社会局","庶務課"]
[26,"社会局","保護課"]
[27,"社会局","福利課"]
[28,"社会局","職業課"]
[29,"保健局","庶務課"]
[30,"保健局","衛生課"]
[31,"保健局","公園課"]
[32,"水道局","庶務課"]
[33,"水道局","会計課"]
[34,"水道局","業務課"]
[35,"水道局","給水課"]
[36,"水道局","拡張課"]
[37,"水道局","下水課"]
[38,"土木局","庶務課"]
[39,"土木局","道路管理課"]
[40,"土木局","道路建設課"]
[41,"土木局","河川課"]
[42,"電気局","総務課"]
[43,"電気局","労働課"]
[44,"電気局","経理課"]
[45,"電気局","会計課"]
[46,"運輸部","庶務課"]
[47,"運輸部","業務課"]
[48,"運輸部","事業普及課"]
[49,"運輸部","車輌課"]
[50,"運輸部","保線課"]
[51,"運輸部","交通統制調査課"]
[52,"電燈部","営業課"]
[53,"電燈部","配線課"]
[54,"電燈部","電力課"]
[55,"監査部","監察部"]
[56,"監査部","区政部"]
[57,"市民動員部","国民総精神総

Type 1 error occured in the following offices.

[[2,"Admin","文書課"],
[10,"経理局","主税課"],
[20,"教育局","庶務課"],
[24,"教育局","視学課"],
[30,"保健局","衛生課"],
[34,"水道局","業務課"],
[37,"水道局","下水課"],
[41,"土木局","河川課"],
[42,"電気局","総務課"],
[63,"港湾部","技術課"],
[64,"港湾部","地所課"],
[67,"中央卸売市場","会計課"],
[73,"養育院","安房分院"],
[79,"小河内貯水池建設事務所","工事課"],
[81,"臨時建築部","第一工營課"]]


In [59]:
Type1=[[2,"Admin","文書課"],
       [10,"経理局","主税課"],
      [20,"教育局","庶務課"],
       [24,"教育局","視学課"],
       [30,"保健局","衛生課"],
       [34,"水道局","業務課"],
       [37,"水道局","下水課"],
      [41,"土木局","河川課"],
       [57,"市民動員部","国民総精神総動員課"],
      [63,"港湾部","技術課"],
      [73,"養育院","安房分院"],
      [79,"小河内貯水池建設事務所","工事課"],
      [82,"臨時建築部","第二工營課"]]

In [46]:
if len(FailedList)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')

Fantastic!! Success Rate is 0.9047619047619048


In [47]:
#Fixing Failed Offices
#Step1: Check for simple page assignment error
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
for n in range(2,len(FailedList)):
    Dept=FailedList[n][1]
    Office=FailedList[n][2]
    print(Dept,Office)
    Page=df['Page'][(df['Office']==Office) & (df['Dept']==Dept)].tolist()[0]
    image=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    cv2.imshow('Image',image)
    cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(0)

教育局 社会教育課
社会局 庶務課
保健局 公園課
水道局 会計課
土木局 道路管理課
市民動員部 国民総精神総動員課


PageSplit Error

港湾部 港務課:p286,p287

=> Fixed

In [48]:
FailedList

[[7, '企画局', '都市計画課'],
 [11, '経理局', '徴収課'],
 [22, '教育局', '社会教育課'],
 [25, '社会局', '庶務課'],
 [31, '保健局', '公園課'],
 [33, '水道局', '会計課'],
 [39, '土木局', '道路管理課'],
 [57, '市民動員部', '国民総精神総動員課']]

In [49]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        #cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        #cv2.imshow('pic',img)
        #cv2.waitKey(0)
    except:
        FailedList2.append(list((n,Dept,Office)))
        continue

都市計画課success: at row 7
徴収課success: at row 11
社会教育課success: at row 22
庶務課success: at row 25
公園課success: at row 31
会計課success: at row 33
道路管理課success: at row 39
国民総精神総動員課success: at row 57


In [50]:
FailedList2

[]

In [26]:
XCoord_Unit

In [52]:
#Fixing Failed Offices
#Step3: Check for simple page assignment error
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
for n in range(2,len(Type1)):
    Dept=Type1[n][1]
    Office=Type1[n][2]
    print(Dept,Office)
    try:
        Page=df['Page'][(df['Office']==Office) & (df['Dept']==Dept)].tolist()[0]
        image=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        cv2.imshow('Image',image)
        cv2.waitKey(0)
    except:
        continue

土木局 河川課
港湾部 技術課
養育院 安房分院
小河内貯水池建設事務所 工事課


In [18]:
Type1

[[2, 'Admin', '文書課'],
 [20, '教育局', '庶務課'],
 [34, '水道局', '業務課'],
 [37, '水道局', '下水課'],
 [42, '電気局', '総務課'],
 [63, '港湾部', '技術課'],
 [64, '港湾部', '地所課'],
 [67, '中央卸売市場', '会計課'],
 [73, '養育院', '安房分院'],
 [79, '小河内貯水池建設事務所', '工事課'],
 [81, '臨時建築部', '第一工營課']]

In [60]:
Type1_2=[]
for Office in Type1:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        cv2.imshow('pic',img)
        cv2.waitKey(0)
    except:
        Type1_2.append(list((n,Dept,Office)))
        continue

文書課success: at row 2
主税課success: at row 10
庶務課success: at row 20
視学課success: at row 24
衛生課success: at row 30
業務課success: at row 34
下水課success: at row 37
河川課success: at row 41
国民総精神総動員課success: at row 57
技術課success: at row 63
安房分院success: at row 73
工事課success: at row 79
第二工營課success: at row 82


In [86]:
Show(81,360)

In [87]:
dta[Year]["市民動員部"]['国民総精神総動員課']={'Starting_Page': 264,
                          'Office_X1':280 ,
                          'Ending_Page':265 ,
                          'Office_X2':282 ,
                          'Page_Range': [264, 265]}

dta[Year]["臨時建築部"]['第一工營課']={'Starting_Page': 311,
 'Office_X1': 360,
 'Ending_Page': 313,
 'Office_X2': 227,
 'Page_Range': [311, 312, 313]}

In [54]:
FailRate=len(FailedList2)/len(df)
if len(FailedList2)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList2)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList2)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate

DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')

Fantastic!! Success Rate is 1.0


OSError: [Errno 22] Invalid argument

In [81]:
Check(df)

[0,"Admin","秘書課"]
[1,"Admin","人事課"]
[2,"Admin","文書課"]
[3,"企画局","庶務課"]
[4,"企画局","企画課"]
[5,"企画局","財務課"]
[6,"企画局","統計課"]
[7,"企画局","都市計画課"]
[8,"経理局","会計課"]
[9,"経理局","公債課"]
[10,"経理局","主税課"]
[11,"経理局","徴収課"]
[12,"経理局","用度課"]
[13,"経理局","地理課"]
[14,"産業局","庶務課"]
[15,"産業局","商工課"]
[16,"産業局","貿易課"]
[17,"産業局","北京出張所"]
[18,"産業局","農漁課"]
[19,"産業局","権度課"]
[20,"教育局","庶務課"]
[21,"教育局","学務課"]
[22,"教育局","社会教育課"]
[23,"教育局","体育課"]
[24,"教育局","視学課"]
[25,"社会局","庶務課"]
[26,"社会局","保護課"]
[27,"社会局","福利課"]
[28,"社会局","職業課"]
[29,"保健局","庶務課"]
[30,"保健局","衛生課"]
[31,"保健局","公園課"]
[32,"水道局","庶務課"]
[33,"水道局","会計課"]
[34,"水道局","業務課"]
[35,"水道局","給水課"]
[36,"水道局","拡張課"]
[37,"水道局","下水課"]
[38,"土木局","庶務課"]
[39,"土木局","道路管理課"]
[40,"土木局","道路建設課"]
[41,"土木局","河川課"]
[42,"電気局","総務課"]
[43,"電気局","労働課"]
[44,"電気局","経理課"]
[45,"電気局","会計課"]
[46,"運輸部","庶務課"]
[47,"運輸部","業務課"]
[48,"運輸部","事業普及課"]
[49,"運輸部","車輌課"]
[50,"運輸部","保線課"]
[51,"運輸部","交通統制調査課"]
[52,"電燈部","営業課"]
[53,"電燈部","配線課"]
[54,"電燈部","電力課"]
[55,"監査部","監察部"]
[56,"監査部","区政部"]
[57,"市民動員部","国民総精神総

In [88]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame_Office.json", "w") as outfile:
    outfile.write(json_object)